# Séance 3 : flexibilité et optimisation

### Imports

In [ ]:
# À exécuter en premier sur Google Colab (sans effet en local) : récupère les données du cours et installe pulp
import os, sys
if "google.colab" in sys.modules:
    DEPOT = "https://github.com/borges67/export_cours_modelisation.git"
    DOSSIER = DEPOT.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(f"/content/{DOSSIER}"):
        !git clone -q --depth 1 {DEPOT} /content/{DOSSIER}
    os.chdir(f"/content/{DOSSIER}")
    !pip install -q "pulp<4"   # PuLP 4 (novembre 2026) change la création des variables : on garde la version 3
print(os.getcwd())

In [ ]:
# Télécharger le dossier data/ sur son ordinateur (Colab uniquement)
import sys
if "google.colab" in sys.modules:
    !zip -r -q /content/data.zip data
    from google.colab import files
    files.download("/content/data.zip")

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
import modele_lineaire as ml   # ml.tracer_semaine : graphique d'une semaine

pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (10, 3.5)

## 1. Le modèle linéaire

Le modèle choisit en même temps les **capacités installées** (solaire en toiture et en ombrière, batterie de 2 h) et
leur fonctionnement heure par heure sur toute l'année 2040, pour **minimiser le coût total** : la facture
d'électricité plus le coût annuel des installations. La batterie se charge avec le solaire ou sur le réseau. Seul
le solaire peut être vendu ; ce qui n'est ni consommé, ni stocké, ni vendu est écrêté. La batterie fait au plus un
cycle par jour, et son stock est le même en fin et en début d'année.

Toutes les données d'entrée viennent des séances 1 et 2 (`data/resultats/`) et des l'interne (Artelys) (`data/interne/`) :
on peut changer de bâtiment, de tarif ou d'années climatiques dans la cellule suivante.

In [ ]:
BATIMENT = "bureaux"                # "bureau_de_poste", "bureaux", "plateforme_tri" ou "plateforme_distribution"
TARIF = "spot"                      # tarif d'achat de la séance 2 : "spot", "plat" ou "hphc"
ANNEES = ["2008", "2009", "2012"]   # années climatiques : les scénarios de la partie 2
ANNEE = "2008"                      # année climatique de la partie 1

# Consommation 2040 du bâtiment (kW), séance 1 : somme des usages, une colonne par année climatique
USAGES = ["ecs", "usages_specifiques", "chauffage_joule", "chauffage_pac", "climatisation", "recharge_ve"]
CONSO = 0
for u in USAGES:
    CONSO = CONSO + pd.read_csv(f"data/resultats/consommation/2040/{BATIMENT}/{u}.csv", index_col="date_heure", parse_dates=True)[ANNEES]

# Production d'un kWc de solaire (kW par kWc), une colonne par année climatique
FACTEUR_PV = pd.read_csv("data/interne/courbe_pv.csv", index_col="date_heure", parse_dates=True)[ANNEES]

# Exports de la séance 2 : tarifs (EUR/MWh), puissances de solaire possibles sur le bâtiment (kWc)
TARIFS = pd.read_csv(f"data/resultats/prix/{BATIMENT}/tarifs.csv", index_col="date_heure", parse_dates=True)
KWC = pd.read_csv(f"data/resultats/prix/{BATIMENT}/kwc.csv", index_col="niveau")

**Les données**, une valeur par heure `t` (cellules suivantes) :

| Nom | Signification | Unité |
|---|---|---|
| `conso[t]` | consommation du site | kW |
| `facteur[t]` | production d'un kWc de solaire | kW par kWc |
| `prix_achat[t]`, `prix_vente[t]` | prix de l'électricité achetée, vendue | EUR/MWh |
| `RENDEMENT` | rendement de la batterie, à la charge comme à la décharge (0,95) | - |
| `STOCK_MIN`, `STOCK_MAX` | stock minimal et maximal, en part de la capacité (10 % et 95 %) | - |
| `ANNUITE_TOITURE`, `ANNUITE_OMBRIERE`, `ANNUITE_BATTERIE` | coût annualisé d'un kWc de toiture, d'un kWc d'ombrière, d'un kWh de batterie | EUR/an |
| `HEURES`, `JOURS`, `FIN` | les heures 0 à 8 759, les jours 0 à 364 ; `FIN` = 8 760 | |

**Les variables de décision**, positives ou nulles, choisies par le solveur :

| Nom | Signification | Unité |
|---|---|---|
| `kwc_toiture`, `kwc_ombriere` | puissance de solaire installée en toiture, en ombrière | kWc |
| `puissance`, `capacite` | puissance de la batterie, et sa capacité (`DUREE_BATTERIE × puissance`) | kW, kWh |
| `achat[t]`, `vente[t]` | puissance achetée au réseau, vendue au réseau | kW |
| `ecretement[t]` | production solaire écrêtée (ni consommée, ni stockée, ni vendue) | kW |
| `charge[t]`, `decharge[t]` | puissance qui entre dans la batterie, qui en sort | kW |
| `stock[t]` | énergie dans la batterie au début de l'heure `t` (`stock[FIN]` : en fin d'année) | kWh |

**En Python** (outils du modèle, définis plus bas) :
- `modele, achat, vente, ecretement, charge, decharge, stock = modele_vide(FIN)` crée le modèle et ses variables heure par heure ;
- `kwc_toiture, kwc_ombriere, puissance, capacite = capacites_vides()` crée les capacités, entre leurs bornes ;
- `modele += a <= b` ajoute une contrainte (de même avec `>=` et `==`) ;
- `modele += expression`, sans comparaison donne l'objectif à minimiser ;
- `somme(x[t] for t in HEURES)` additionne sur toutes les heures ;
- `resultat, capacites, cout_total = resoudre(modele, annee)` résout le modèle et renvoie les résultats heure par heure,
  les capacités choisies et la valeur de l'objectif.

Au pas horaire, x kW pendant une heure font x kWh : on additionne directement des kW et des kWh.

In [ ]:
# Prix, une valeur par heure (la consommation et le solaire sont lus dans la cellule du modèle)
prix_achat = TARIFS[TARIF]  # EUR/MWh
prix_vente = prix_achat - 5  # EUR/MWh 
PRIX = pd.DataFrame({"prix_achat": prix_achat, "prix_vente": prix_vente}, index=CONSO.index)   # enregistrés en partie 2

# Batterie (HYPOTHÈSES PÉDAGOGIQUES)
DUREE_BATTERIE = 2    # h : capacité (kWh) = 2 × puissance (kW)
RENDEMENT = 0.95      # à la charge comme à la décharge
STOCK_MIN = 0.10      # stock minimal et maximal, en part de la capacité
STOCK_MAX = 0.95

# Bornes des capacités 
TOITURE_MIN, TOITURE_MAX = ... , ...     # kWc
OMBRIERE_MIN, OMBRIERE_MAX = ... , ...   # kWc
BATTERIE_MIN, BATTERIE_MAX = ... , ...        # kW

FIN = len(CONSO)            # 8 760 heures
HEURES = range(FIN)         # 0, 1, ..., 8 759
JOURS = range(FIN // 24)    # 0, 1, ..., 364

# Coûts annualisés
ANNUITE_TOITURE = ...
ANNUITE_OMBRIERE = ...
ANNUITE_BATTERIE = ...

### Outils du modèle

Les deux cellules suivantes créent les variables du modèle, puis le résolvent avec le solveur HiGHS et rangent les
résultats heure par heure. Pas besoin de les modifier.

In [ ]:
somme = pulp.lpSum   # somme(x[t] for t in HEURES) : somme d'expressions du modèle


def variables(nom, nombre):
    """Liste de variables positives ou nulles : nom_0, nom_1, ..."""
    return [pulp.LpVariable(f"{nom}_{t}", lowBound=0) for t in range(nombre)]


def modele_vide(nb_heures):
    """Modèle à minimiser et ses variables, une par heure (le stock en a une de plus : la fin d'année)."""
    modele = pulp.LpProblem("site", pulp.LpMinimize)
    return (modele, variables("achat", nb_heures), variables("vente", nb_heures), variables("ecretement", nb_heures),
            variables("charge", nb_heures), variables("decharge", nb_heures), variables("stock", nb_heures + 1))


def capacites_vides():
    """Capacités à installer, entre leurs bornes : kWc de toiture et d'ombrière, puissance de la batterie (kW) et sa
    capacité (kWh), qui vaut DUREE_BATTERIE × puissance."""
    kwc_toiture = pulp.LpVariable("kwc_toiture", lowBound=TOITURE_MIN, upBound=TOITURE_MAX)
    kwc_ombriere = pulp.LpVariable("kwc_ombriere", lowBound=OMBRIERE_MIN, upBound=OMBRIERE_MAX)
    puissance = pulp.LpVariable("puissance_batterie", lowBound=BATTERIE_MIN, upBound=BATTERIE_MAX)
    return kwc_toiture, kwc_ombriere, puissance, DUREE_BATTERIE * puissance

In [ ]:
def resoudre(modele, annee):
    """Résout le modèle de l'année climatique `annee`. Renvoie les résultats heure par heure (kW, stock en kWh), les
    capacités choisies et la valeur de l'objectif (EUR/an). Une variable absente du modèle vaut 0."""
    solveur = pulp.HiGHS(msg=False) if pulp.HiGHS().available() else pulp.PULP_CBC_CMD(msg=False)
    modele.solve(solveur)
    if pulp.LpStatus[modele.status] != "Optimal":
        raise RuntimeError(f"le modèle n'a pas de solution optimale ({pulp.LpStatus[modele.status]})")

    # Valeur de chaque variable (le solveur peut renvoyer -0,000001 au lieu de 0)
    valeurs = {v.name: max(0, v.varValue) for v in modele.variables()}

    capacites = pd.Series({"toiture_kwc": valeurs.get("kwc_toiture", 0), "ombriere_kwc": valeurs.get("kwc_ombriere", 0),
                           "batterie_kw": valeurs.get("puissance_batterie", 0)}, dtype=float)
    capacites["batterie_kwh"] = DUREE_BATTERIE * capacites["batterie_kw"]

    heures = range(len(CONSO))
    resultat = pd.DataFrame(index=CONSO.index)
    resultat["consommation"] = CONSO[annee].values
    resultat["production_pv"] = FACTEUR_PV[annee].values * (capacites["toiture_kwc"] + capacites["ombriere_kwc"])
    resultat["ecretement"] = [valeurs.get(f"ecretement_{t}", 0.0) for t in heures]
    resultat["charge_batterie"] = [valeurs.get(f"charge_{t}", 0.0) for t in heures]
    resultat["decharge_batterie"] = [valeurs.get(f"decharge_{t}", 0.0) for t in heures]
    resultat["stock_batterie"] = [valeurs.get(f"stock_{t}", 0.0) for t in heures]
    resultat["achat_reseau"] = [valeurs.get(f"achat_{t}", 0.0) for t in heures]
    resultat["vente_reseau"] = [valeurs.get(f"vente_{t}", 0.0) for t in heures]
    return resultat, capacites, pulp.value(modele.objective)

### Écrire le modèle

In [ ]:
# Données de l'année climatique de la partie 1, une valeur par heure. Elles sont lues ici, et pas plus haut, car la
# boucle de la partie 2 remplace conso et facteur par ceux de la dernière année climatique
conso = CONSO[ANNEE].values               # kW
facteur = FACTEUR_PV[ANNEE].values        # kW par kWc

modele, achat, vente, ecretement, charge, decharge, stock = modele_vide(FIN)
kwc_toiture, kwc_ombriere, puissance, capacite = capacites_vides()
for t in HEURES:
    modele += ... == ...      #


# Écrivez l'objectif (9) : la facture en EUR, plus le coût annualisé des installations
modele += somme(... for t in HEURES) / 1000 + ...                         

In [ ]:
# Résultats et indicateurs à calculer eux-même d'après les résultats du modèle

SEMAINE = "2023-05-25"      # premier jour de la semaine tracée
resultat, capacites, cout_total = resoudre(modele, ANNEE)
print(f"Coût total : {cout_total:,.0f} EUR/an")
print(capacites.round(1))
ml.tracer_semaine(resultat, prix_achat, prix_vente, SEMAINE, f"{BATIMENT}, année climatique {ANNEE}")

In [ ]:
# Calculer sur d'autres scénarios

# Avec / Sans injection
# Prix de la batterie
# Autres courbe de prix 
# Minimum d'installation

## 2. Scénarisation

On fait tourner le modèle de la partie 1 pour les **3 années climatiques** de la séance 2 (2008, 2009, 2012), avec un
seul tarif. Les résultats sont enregistrés dans `data/resultats/modele/<bâtiment>/` :
- un fichier par actif : `consommation.csv`, `production_pv.csv`, `ecretement.csv`, `charge_batterie.csv`,
  `decharge_batterie.csv`, `stock_batterie.csv`, `achat_reseau.csv`, `vente_reseau.csv` ; une ligne par heure, une
  colonne par année climatique, en kW (le stock en kWh) ;
- `capacites.csv` : les capacités choisies pour chaque année climatique ;
- `prix.csv` : les prix d'achat et de vente utilisés (EUR/MWh).

In [ ]:
RESULTATS = {}   # résultats heure par heure de chaque année climatique
CAPACITES = {}   # capacités choisies pour chaque année climatique

for annee in ANNEES:
    # Données de l'année climatique
    conso = CONSO[annee].values
    facteur = FACTEUR_PV[annee].values

    # Recopiez ici votre modèle de la partie 1
    ...

    RESULTATS[annee], CAPACITES[annee], cout_total = resoudre(modele, annee)
    print(f"{annee} : coût total {cout_total:,.0f} EUR/an ; toiture {CAPACITES[annee]['toiture_kwc']:.0f} kWc, "
          f"ombrière {CAPACITES[annee]['ombriere_kwc']:.0f} kWc, batterie {CAPACITES[annee]['batterie_kw']:.0f} kW")

In [ ]:
# Création du dossier de résultat
dossier = f"data/resultats/modele/{BATIMENT}"
os.makedirs(dossier, exist_ok=True)

ACTIFS = ["consommation", "production_pv", "ecretement", "charge_batterie", "decharge_batterie",
          "stock_batterie", "achat_reseau", "vente_reseau"]
for actif in ACTIFS:
    courbes = pd.DataFrame({annee: RESULTATS[annee][actif] for annee in ANNEES})
    courbes.round(3).to_csv(f"{dossier}/{actif}.csv", index_label="date_heure")
pd.DataFrame(CAPACITES).round(2).to_csv(f"{dossier}/capacites.csv", index_label="capacite")
PRIX.round(3).to_csv(f"{dossier}/prix.csv", index_label="date_heure")

print("Résultats enregistrés dans", dossier)
pd.DataFrame(CAPACITES).round(1)

In [ ]:
# Tester d'autres scénarios

# Prix de la batterie différent
# Injection autorisée / non autorisée

## 3. Émissions de CO2 : contenu constant ou contenu horaire

On compte les émissions des achats au réseau de deux façons : avec un contenu CO2 **constant** (la moyenne de
l'année), ou avec le contenu **horaire** publié par RTE (éCO2mix 2023, contenu moyen,
`data/interne/courbe_co2.csv`). Le solaire produit sur place est compté à zéro. On compare le bâtiment sans
installation (il achète toute sa consommation) et le bâtiment avec l'installation choisie par le modèle, à partir des
fichiers de `data/resultats/modele/<bâtiment>/`.

In [ ]:
dossier = f"data/resultats/modele/{BATIMENT}"
consommation = pd.read_csv(f"{dossier}/consommation.csv", index_col="date_heure", parse_dates=True)
achat_reseau = pd.read_csv(f"{dossier}/achat_reseau.csv", index_col="date_heure", parse_dates=True)

CO2 = pd.read_csv("data/interne/courbe_co2.csv", index_col="date_heure", parse_dates=True)["co2_g_par_kwh"]
CO2_CONSTANT = CO2.mean()   # gCO2/kWh, moyenne de l'année
print(f"Contenu CO2 moyen : {CO2_CONSTANT:.1f} gCO2/kWh (de {CO2.min():.0f} à {CO2.max():.0f} selon l'heure)")

EMISSIONS = pd.DataFrame(dtype=float)
for annee in consommation.columns:
    for cas, achats in [("sans installation", consommation[annee]), ("avec installation", achat_reseau[annee])]:
        ligne = f"{annee}, {cas}"
        # Calculez les émissions des achats (tCO2/an) avec le contenu constant, puis avec le contenu horaire
        EMISSIONS.loc[ligne, "contenu constant (tCO2/an)"] = ...
        EMISSIONS.loc[ligne, "contenu horaire (tCO2/an)"] = ...
EMISSIONS["écart (%)"] = 100 * (EMISSIONS["contenu horaire (tCO2/an)"] / EMISSIONS["contenu constant (tCO2/an)"] - 1)
EMISSIONS.round(2)

In [ ]:
ax = EMISSIONS[["contenu constant (tCO2/an)", "contenu horaire (tCO2/an)"]].plot.bar(figsize=(10, 4), rot=0)
ax.set_xticklabels([ligne.replace(", ", "\n") for ligne in EMISSIONS.index], fontsize=8)
ax.set_ylabel("tCO2/an")
ax.set_title(f"{BATIMENT} : émissions des achats au réseau selon la façon de compter")
ax.grid(alpha=0.3, axis="y")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

Les deux méthodes donnent-elles les mêmes émissions, et le même gain pour l'installation ?

**Réponse** :

## 4. Investissements et indicateurs

Les indicateurs se calculent à partir des fichiers de `data/resultats/modele/<bâtiment>/` seulement, sans les
variables du modèle : relisez les courbes, puis calculez pour chaque année climatique :
- la consommation, la production solaire, les achats et les ventes (MWh/an), le taux d'autoconsommation ;
- la pointe soutirée au réseau (kW), qui fixe le segment et la part puissance du TURPE ;
- la facture (EUR/an), et celle d'un bâtiment sans solaire ni batterie (tout est acheté) ;
- l'investissement (EUR) et le coût annualisé des installations (EUR/an) ;
- l'économie sur la facture, le gain net (économie moins coût annualisé) et le temps de retour simple (ans).

In [ ]:
dossier = f"data/resultats/modele/{BATIMENT}"
consommation = pd.read_csv(f"{dossier}/consommation.csv", index_col="date_heure", parse_dates=True)   # exemple

# Lisez de même la production solaire, l'écrêtement, les achats et les ventes au réseau, les prix et les capacités
production_pv = ...
ecretement = ...
achat_reseau = ...
vente_reseau = ...
prix = ...
capacites = ...

In [ ]:
INDICATEURS = pd.DataFrame(columns=consommation.columns, dtype=float)
for annee in consommation.columns:
    INDICATEURS.loc["consommation (MWh/an)", annee] = consommation[annee].sum() / 1000   # exemple
    # Calculez les autres indicateurs de l'année : production solaire, taux d'autoconsommation, achats et ventes,
    # pointe soutirée, facture et facture sans solaire ni batterie, économie, investissement, coût annualisé,
    # gain net, temps de retour simple
    ...
INDICATEURS.round(1)

Les capacités choisies changent-elles d'une année climatique à l'autre ? Quelle installation conseiller à LPI ?
Refaites tourner le modèle avec un autre tarif, ou avec la batterie au coût bas : qu'est-ce qui change ?

**Réponse** :